# BioRob Phase 2B — Column Guard + Label-Only Export

Final all-subject version. It reads only existing Phase 2A label CSVs and therefore skips missing Phase 1C files automatically. It does not create extra columns.

In [1]:
# ============================================================
# CELL 1 — Imports and final Phase 2B configuration
# Purpose:
#   Phase 2B reads ONLY existing Phase 2A label files and exports
#   active-only labelonly CSVs for downstream phases.
#
# Important:
#   - It does NOT read raw Phase 1B files.
#   - It does NOT read missing Phase 1C files.
#   - It does NOT create new columns.
#   - It does NOT delete source files.
# ============================================================

from __future__ import annotations

from pathlib import Path
from datetime import datetime
import re
import shutil
import traceback

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

ROOT_DIR = Path("/home/tsultan1/BioRob/Human Subject Data")

SYNC_SUBPATH = Path("cleaned/synchronized_proper_lite_union_v3")
SRC_LABEL_DIRNAME = "label"
DEST_LABEL_DIRNAME = "labelonly"

LABEL_GLOB = "*_icml_consensus_labels.csv"

# Final-run switches
RUN_INPUT_AUDIT = True
RUN_ALL_SUBJECTS = True

# True = regenerate labelonly files if they already exist.
# False = skip existing labelonly outputs.
OVERWRITE_LABELONLY = True

# Copy sidecar onset JSONs if present. This does not affect CSV columns.
COPY_SIDECARS = True

# Keep empty non-T0 outputs for traceability if a non-rest file has zero active rows.
# Recommended False: save empty CSV with correct header.
SKIP_EMPTY_NONREST_OUTPUT = False

AUDIT_DIR = ROOT_DIR / "_audit_phase2B_labelonly"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT_DIR:", ROOT_DIR)
print("Input label path pattern:")
print(ROOT_DIR / "Sub-*" / SYNC_SUBPATH / SRC_LABEL_DIRNAME / LABEL_GLOB)
print("Output folder per subject:")
print(ROOT_DIR / "Sub-*" / SYNC_SUBPATH / DEST_LABEL_DIRNAME)
print("\nRUN_INPUT_AUDIT     =", RUN_INPUT_AUDIT)
print("RUN_ALL_SUBJECTS    =", RUN_ALL_SUBJECTS)
print("OVERWRITE_LABELONLY =", OVERWRITE_LABELONLY)
print("COPY_SIDECARS       =", COPY_SIDECARS)
print("AUDIT_DIR           =", AUDIT_DIR)


ROOT_DIR: /home/tsultan1/BioRob/Human Subject Data
Input label path pattern:
/home/tsultan1/BioRob/Human Subject Data/Sub-*/cleaned/synchronized_proper_lite_union_v3/label/*_icml_consensus_labels.csv
Output folder per subject:
/home/tsultan1/BioRob/Human Subject Data/Sub-*/cleaned/synchronized_proper_lite_union_v3/labelonly

RUN_INPUT_AUDIT     = True
RUN_ALL_SUBJECTS    = True
OVERWRITE_LABELONLY = True
COPY_SIDECARS       = True
AUDIT_DIR           = /home/tsultan1/BioRob/Human Subject Data/_audit_phase2B_labelonly


In [2]:
# ============================================================
# CELL 2 — Exact expected Phase 2A label schema
# Purpose:
#   Use the exact 45 columns observed after Phase 2A.
#
# Important:
#   We do NOT create missing columns.
#   If a file has missing/extra columns, it is skipped and logged.
# ============================================================

EXPECTED_COLUMNS = [
    "subject_id",
    "task",
    "trial",
    "Timestamp_seconds",
    "Timestamp_ms",

    "EMG_Ch1",
    "EMG_Ch2",
    "EMG_Ch3",
    "EMG_Ch4",

    "EEG_Ch1",
    "EEG_Ch2",
    "EEG_Ch3",
    "EEG_Ch4",
    "EEG_Ch5",
    "EEG_Ch6",
    "EEG_Ch7",
    "EEG_Ch8",

    "ET_GazeLeftx",
    "ET_GazeLefty",
    "ET_GazeRightx",
    "ET_GazeRighty",

    "ET_PupilLeft",
    "ET_PupilRight",

    "ET_ValidityLeftEye",
    "ET_ValidityRightEye",
    "ET_Blink",
    "ET_Fixation",
    "ET_Worn",

    "ET_DistanceLeft",
    "ET_DistanceRight",

    "ET_GyroX",
    "ET_GyroY",
    "ET_GyroZ",

    "ET_AccX",
    "ET_AccY",
    "ET_AccZ",

    "ET_HeadRotationPitch",
    "ET_HeadRotationYaw",
    "ET_HeadRotationRoll",

    "active_raw",
    "active",
    "active_prob",
    "label_action",
    "label_11",
    "task_target",
]

EXPECTED_SET = set(EXPECTED_COLUMNS)

META_COLS = ["subject_id", "task", "trial"]
TIME_COLS = ["Timestamp_seconds", "Timestamp_ms"]
LABEL_COLS = ["active_raw", "active", "active_prob", "label_action", "label_11", "task_target"]

EEG_COLS = [f"EEG_Ch{i}" for i in range(1, 9)]
EMG_COLS = [f"EMG_Ch{i}" for i in range(1, 5)]
ET_CORE_COLS = [
    "ET_GazeLeftx", "ET_GazeLefty", "ET_GazeRightx", "ET_GazeRighty",
    "ET_PupilLeft", "ET_PupilRight",
    "ET_ValidityLeftEye", "ET_ValidityRightEye",
    "ET_Blink", "ET_Fixation", "ET_Worn",
]

print("Expected Phase 2A label columns:", len(EXPECTED_COLUMNS))
for i, c in enumerate(EXPECTED_COLUMNS, start=1):
    print(f"{i:02d}. {c}")


Expected Phase 2A label columns: 45
01. subject_id
02. task
03. trial
04. Timestamp_seconds
05. Timestamp_ms
06. EMG_Ch1
07. EMG_Ch2
08. EMG_Ch3
09. EMG_Ch4
10. EEG_Ch1
11. EEG_Ch2
12. EEG_Ch3
13. EEG_Ch4
14. EEG_Ch5
15. EEG_Ch6
16. EEG_Ch7
17. EEG_Ch8
18. ET_GazeLeftx
19. ET_GazeLefty
20. ET_GazeRightx
21. ET_GazeRighty
22. ET_PupilLeft
23. ET_PupilRight
24. ET_ValidityLeftEye
25. ET_ValidityRightEye
26. ET_Blink
27. ET_Fixation
28. ET_Worn
29. ET_DistanceLeft
30. ET_DistanceRight
31. ET_GyroX
32. ET_GyroY
33. ET_GyroZ
34. ET_AccX
35. ET_AccY
36. ET_AccZ
37. ET_HeadRotationPitch
38. ET_HeadRotationYaw
39. ET_HeadRotationRoll
40. active_raw
41. active
42. active_prob
43. label_action
44. label_11
45. task_target


In [3]:
# ============================================================
# CELL 3 — Helper functions for subject/file parsing and validation
# Purpose:
#   Check subject_id, task, trial, columns, and time grid carefully.
# ============================================================

def subject_sort_key(path: Path):
    m = re.search(r"Sub-(\d+)$", path.name, flags=re.I)
    return int(m.group(1)) if m else 10**9


def get_subject_id_from_path(path: Path):
    """Find Sub-N in any part of the path."""
    for part in path.parts:
        m = re.fullmatch(r"Sub-(\d+)", part, flags=re.I)
        if m:
            return int(m.group(1))
    return None


def parse_task_trial_from_filename(path: Path):
    """
    Parse T/M task and trial using BioRob convention:
      FIRST digit after T/M = task
      remaining digits = trial

    Examples:
      003_T515_synchronized_corrected_icml_consensus_labels.csv -> task=5, trial=15
      001_T03_synchronized_corrected_icml_consensus_labels.csv  -> task=0, trial=3
      071_T112_synchronized_corrected_icml_consensus_labels.csv -> task=1, trial=12
    """
    stem = path.stem
    m = re.search(r"(?:^|_)(T|M)(\d{2,})", stem, flags=re.I)
    if not m:
        return None, None, None

    mode_char = m.group(1).upper()
    digits = m.group(2)

    task = int(digits[0])
    trial = int(digits[1:]) if len(digits) > 1 else None

    return mode_char, task, trial


def find_phase2a_label_files():
    """Find label files only from existing Phase 2A outputs. This automatically skips missing Phase 1C files."""
    subject_dirs = sorted(
        [p for p in ROOT_DIR.glob("Sub-*") if p.is_dir()],
        key=subject_sort_key,
    )

    label_files = []
    subject_label_dirs = []

    for sub_dir in subject_dirs:
        label_dir = sub_dir / SYNC_SUBPATH / SRC_LABEL_DIRNAME
        if label_dir.exists():
            files = sorted(label_dir.glob(LABEL_GLOB))
            if files:
                subject_label_dirs.append(label_dir)
                label_files.extend(files)

    return subject_label_dirs, label_files


def find_phase1c_sync_files():
    """Used only for audit/count comparison. Phase 2B does not read these directly."""
    subject_dirs = sorted(
        [p for p in ROOT_DIR.glob("Sub-*") if p.is_dir()],
        key=subject_sort_key,
    )

    sync_files = []
    for sub_dir in subject_dirs:
        sync_dir = sub_dir / SYNC_SUBPATH
        if sync_dir.exists():
            sync_files.extend(sorted(sync_dir.glob("*_synchronized_corrected.csv")))

    return sync_files


def read_header_only(csv_path: Path):
    return list(pd.read_csv(csv_path, nrows=0).columns)


def safe_unique_int(series: pd.Series):
    vals = pd.to_numeric(series, errors="coerce").dropna().unique()
    if len(vals) != 1:
        return None, vals
    return int(vals[0]), vals


def estimate_fs_from_time(df: pd.DataFrame):
    t = pd.to_numeric(df["Timestamp_seconds"], errors="coerce").to_numpy(dtype=float)
    t = t[np.isfinite(t)]

    if len(t) < 3:
        return np.nan, np.nan

    dt = np.diff(t)
    dt = dt[np.isfinite(dt)]
    dt = dt[dt > 0]

    if len(dt) == 0:
        return np.nan, np.nan

    med_dt = float(np.median(dt))
    fs = 1.0 / med_dt if med_dt > 0 else np.nan
    return fs, med_dt


def validate_phase2a_label_csv(csv_path: Path, load_full: bool = True):
    """
    Validate one Phase 2A label CSV.
    This function does not modify the file and does not add any column.
    """
    row = {
        "file": str(csv_path),
        "subject": None,
        "filename": csv_path.name,
        "status": "OK",
        "errors": "",
        "warnings": "",
        "n_rows": None,
        "n_cols": None,
        "subject_id": None,
        "task": None,
        "trial": None,
        "filename_mode": None,
        "filename_task": None,
        "filename_trial": None,
        "fs_est": None,
        "median_dt": None,
        "active_rows": None,
        "rest_rows": None,
        "is_rest_task0": None,
    }

    errors = []
    warnings = []

    subject_from_path = get_subject_id_from_path(csv_path)
    row["subject"] = f"Sub-{subject_from_path}" if subject_from_path is not None else None

    mode_char, file_task, file_trial = parse_task_trial_from_filename(csv_path)
    row["filename_mode"] = mode_char
    row["filename_task"] = file_task
    row["filename_trial"] = file_trial
    row["is_rest_task0"] = bool(file_task == 0) if file_task is not None else None

    try:
        header = read_header_only(csv_path)
    except Exception as e:
        errors.append(f"Could not read header: {e}")
        row["status"] = "ERROR"
        row["errors"] = " | ".join(errors)
        return row

    row["n_cols"] = len(header)

    duplicated = sorted([c for c in set(header) if header.count(c) > 1])
    if duplicated:
        errors.append(f"Duplicate columns: {duplicated}")

    missing = [c for c in EXPECTED_COLUMNS if c not in header]
    extra = [c for c in header if c not in EXPECTED_SET]

    if missing:
        errors.append(f"Missing columns: {missing}")
    if extra:
        errors.append(f"Extra columns: {extra}")

    # Do not fail just because order differs, but log it.
    # The export step will reorder to EXPECTED_COLUMNS using only existing columns.
    if not missing and not extra and header != EXPECTED_COLUMNS:
        warnings.append("Column order differs from expected; will be reordered in labelonly output")

    if mode_char is None or file_task is None or file_trial is None:
        errors.append("Could not parse task/trial from filename")

    if not load_full:
        row["status"] = "ERROR" if errors else "OK"
        row["errors"] = " | ".join(errors)
        row["warnings"] = " | ".join(warnings)
        return row

    try:
        df = pd.read_csv(csv_path, low_memory=False)
    except Exception as e:
        errors.append(f"Could not read CSV: {e}")
        row["status"] = "ERROR"
        row["errors"] = " | ".join(errors)
        row["warnings"] = " | ".join(warnings)
        return row

    row["n_rows"] = len(df)

    if len(df) == 0:
        warnings.append("CSV has zero rows")

    # Metadata checks
    for c in META_COLS:
        if c not in df.columns:
            errors.append(f"Missing metadata column after load: {c}")

    if not errors:
        sub_val, sub_unique = safe_unique_int(df["subject_id"])
        task_val, task_unique = safe_unique_int(df["task"])
        trial_val, trial_unique = safe_unique_int(df["trial"])

        row["subject_id"] = sub_val
        row["task"] = task_val
        row["trial"] = trial_val

        if sub_val is None:
            errors.append(f"subject_id is not constant: {sub_unique}")
        elif subject_from_path is not None and sub_val != subject_from_path:
            errors.append(f"subject_id mismatch: CSV={sub_val}, folder=Sub-{subject_from_path}")

        if task_val is None:
            errors.append(f"task is not constant: {task_unique}")
        elif file_task is not None and task_val != file_task:
            errors.append(f"task mismatch: CSV={task_val}, filename={file_task}")

        if trial_val is None:
            errors.append(f"trial is not constant: {trial_unique}")
        elif file_trial is not None and trial_val != file_trial:
            errors.append(f"trial mismatch: CSV={trial_val}, filename={file_trial}")

    # Time grid check
    if "Timestamp_seconds" in df.columns and len(df) > 2:
        fs_est, med_dt = estimate_fs_from_time(df)
        row["fs_est"] = fs_est
        row["median_dt"] = med_dt

        if not np.isfinite(fs_est):
            errors.append("Could not estimate sampling rate from Timestamp_seconds")
        elif abs(fs_est - 250.0) > 2.0:
            warnings.append(f"Estimated fs not close to 250 Hz: {fs_est:.3f}")

    # Label checks
    if "active" in df.columns:
        active_num = pd.to_numeric(df["active"], errors="coerce")
        bad_active = active_num.isna().sum()
        if bad_active > 0:
            errors.append(f"active has non-numeric/NaN values: {bad_active}")

        active_bin = active_num.fillna(0).astype(int)
        unique_active = sorted(active_bin.unique().tolist())
        if not set(unique_active).issubset({0, 1}):
            errors.append(f"active contains values outside 0/1: {unique_active}")

        row["active_rows"] = int((active_bin == 1).sum())
        row["rest_rows"] = int((active_bin == 0).sum())

    if "label_action" in df.columns:
        la = pd.to_numeric(df["label_action"], errors="coerce").fillna(0).astype(int)
        unique_la = sorted(la.unique().tolist())
        if not set(unique_la).issubset({0, 1}):
            errors.append(f"label_action contains values outside 0/1: {unique_la}")

    if "active" in df.columns and "label_action" in df.columns:
        active_num = pd.to_numeric(df["active"], errors="coerce").fillna(0).astype(int)
        label_num = pd.to_numeric(df["label_action"], errors="coerce").fillna(0).astype(int)
        mismatch = int((active_num != label_num).sum())
        if mismatch > 0:
            warnings.append(f"active and label_action differ in {mismatch} rows")

    # For T0/REST files, labels should be rest-only. Warn, do not edit.
    if file_task == 0 and "label_action" in df.columns:
        la_sum = int(pd.to_numeric(df["label_action"], errors="coerce").fillna(0).sum())
        if la_sum != 0:
            warnings.append(f"T0 file has label_action sum={la_sum}; expected 0")

    row["status"] = "ERROR" if errors else "OK"
    row["errors"] = " | ".join(errors)
    row["warnings"] = " | ".join(warnings)
    return row


print("Helper functions loaded.")


Helper functions loaded.


In [ ]:
# ============================================================
# CELL 4 — Input audit: use only existing Phase 2A label files
# Purpose:
#   Confirm Phase 2B will automatically skip the 104 missing Phase 1C files.
# ============================================================

subject_label_dirs, label_files = find_phase2a_label_files()
sync_files = find_phase1c_sync_files()

print("=" * 100)
print("PHASE 2B INPUT DISCOVERY")
print("=" * 100)
print("Found subject label dirs:", len(subject_label_dirs))
for d in subject_label_dirs:
    n = len(sorted(d.glob(LABEL_GLOB)))
    print(f"{d}: {n} label files")

print("\nTotal Phase 1C synchronized files:", len(sync_files))
print("Total Phase 2A label files       :", len(label_files))

# Missing Phase 1C exclusion file from previous audit, if available.
phase1c_exclusion_file = ROOT_DIR / "_audit_phase1c_exclusions" / "excluded_missing_phase1c_files.csv"
if phase1c_exclusion_file.exists():
    phase1c_excluded_df = pd.read_csv(phase1c_exclusion_file)
    print("\nPhase 1C missing/excluded file list found:")
    print(phase1c_exclusion_file)
    print("Excluded missing Phase 1C files:", len(phase1c_excluded_df))
else:
    print("\nNo Phase 1C exclusion file found. That is okay if you already audited separately.")

# Save the exact file list used by Phase 2B.
phase2b_input_df = pd.DataFrame({
    "label_csv": [str(p) for p in label_files],
    "subject": [f"Sub-{get_subject_id_from_path(p)}" for p in label_files],
    "filename": [p.name for p in label_files],
})

input_list_path = AUDIT_DIR / "phase2B_input_label_files.csv"
phase2b_input_df.to_csv(input_list_path, index=False)

print("\nSaved Phase 2B input file list:")
print(input_list_path)

if len(label_files) == 0:
    raise RuntimeError("No Phase 2A label files found. Stop here.")

if len(label_files) < len(sync_files):
    print("\n⚠️ Some synchronized files do not have Phase 2A labels yet.")
    print("   Finish Phase 2A first, or Phase 2B will only use existing labels.")
elif len(label_files) == len(sync_files):
    print("\n✅ Phase 2A labels match existing Phase 1C synchronized files.")
else:
    print("\n⚠️ Label file count is greater than sync file count. Check for duplicates.")

print("\n✅ Phase 2B will read only existing label files.")
print("✅ Missing Phase 1C files cannot enter Phase 2B because they have no synchronized/label CSV.")


In [ ]:
# ============================================================
# CELL 5 — Full Phase 2A label CSV audit
# Purpose:
#   Validate columns, subject_id, task, trial, labels, and 250 Hz grid.
# ============================================================

audit_rows = []

if RUN_INPUT_AUDIT:
    print("=" * 100)
    print("RUNNING PHASE 2B INPUT AUDIT")
    print("=" * 100)

    for i, f in enumerate(label_files, start=1):
        row = validate_phase2a_label_csv(f, load_full=True)
        audit_rows.append(row)

        if row["status"] != "OK":
            print(f"❌ [{i}/{len(label_files)}] {f.name} | {row['errors']}")
        elif row["warnings"]:
            print(f"⚠️ [{i}/{len(label_files)}] {f.name} | {row['warnings']}")
        elif i <= 5:
            print(f"✅ [{i}/{len(label_files)}] {f.name} | rows={row['n_rows']} | active_rows={row['active_rows']}")

    audit_df = pd.DataFrame(audit_rows)

    audit_path = AUDIT_DIR / "phase2B_input_audit.csv"
    audit_df.to_csv(audit_path, index=False)

    print("\n" + "=" * 100)
    print("PHASE 2B INPUT AUDIT SUMMARY")
    print("=" * 100)
    print(audit_df["status"].value_counts(dropna=False))
    print("\nSaved:", audit_path)

    bad_df = audit_df[audit_df["status"] != "OK"].copy()
    if len(bad_df) > 0:
        bad_path = AUDIT_DIR / "phase2B_input_invalid_files.csv"
        bad_df.to_csv(bad_path, index=False)
        print("\n❌ Invalid files found. Saved:")
        print(bad_path)
        display(bad_df[["subject", "filename", "errors"]].head(30))
        raise RuntimeError("Stop: invalid Phase 2A label files found. Fix before Phase 2B export.")
    else:
        print("\n✅ All Phase 2A label files passed strict Phase 2B input checks.")

    subject_counts = (
        audit_df.groupby("subject")["filename"]
        .count()
        .reset_index(name="phase2A_label_files")
        .sort_values("subject")
    )
    subject_counts_path = AUDIT_DIR / "phase2B_input_counts_by_subject.csv"
    subject_counts.to_csv(subject_counts_path, index=False)

    print("\nLabel files by subject:")
    display(subject_counts)

else:
    print("RUN_INPUT_AUDIT=False, skipping full input audit.")
    audit_df = pd.DataFrame()


In [6]:
# ============================================================
# CELL 6 — Phase 2B export function
# Purpose:
#   - T0 / REST files: copy full labels without active filtering.
#   - Non-T0 files: export only active==1 rows.
#   - Save to labelonly/ folder.
#
# Important:
#   No source file is modified.
#   No columns are created.
# ============================================================

def destination_for_labelonly(src_csv: Path):
    # src: .../synchronized_proper_lite_union_v3/label/file.csv
    # dst: .../synchronized_proper_lite_union_v3/labelonly/file.csv
    sync_dir = src_csv.parent.parent
    dest_dir = sync_dir / DEST_LABEL_DIRNAME
    dest_dir.mkdir(parents=True, exist_ok=True)
    return dest_dir / src_csv.name


def sidecars_for_label(src_csv: Path):
    """
    Copy useful sidecars if present. This does not affect CSV data.
    Phase 2A usually creates *_onsets.json.
    """
    candidates = []

    # Example:
    # 003_T515_synchronized_corrected_icml_consensus_labels.csv
    # -> 003_T515_synchronized_corrected_onsets.json
    onsets_name = src_csv.stem.replace("_icml_consensus_labels", "_onsets") + ".json"
    candidates.append(src_csv.with_name(onsets_name))

    # Keep this optional older sidecar pattern if it exists.
    candidates.append(src_csv.with_suffix(".unitconv.json"))

    # Remove duplicates while preserving order.
    seen = set()
    out = []
    for p in candidates:
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


def export_one_phase2b_labelonly(src_csv: Path):
    result = {
        "src_csv": str(src_csv),
        "dest_csv": "",
        "subject": f"Sub-{get_subject_id_from_path(src_csv)}",
        "filename": src_csv.name,
        "status": "OK",
        "error": "",
        "warning": "",
        "task": None,
        "trial": None,
        "is_rest_task0": None,
        "rows_before": None,
        "rows_after": None,
        "rows_removed": None,
        "active_rows_before": None,
        "mode": None,
        "reordered_columns": False,
        "sidecars_copied": 0,
    }

    # Strict validation before export.
    check = validate_phase2a_label_csv(src_csv, load_full=True)
    if check["status"] != "OK":
        result["status"] = "SKIP_INVALID"
        result["error"] = check["errors"]
        result["warning"] = check["warnings"]
        return result

    mode_char, file_task, file_trial = parse_task_trial_from_filename(src_csv)
    result["mode"] = mode_char
    result["task"] = file_task
    result["trial"] = file_trial
    result["is_rest_task0"] = bool(file_task == 0)

    dest_csv = destination_for_labelonly(src_csv)
    result["dest_csv"] = str(dest_csv)

    if dest_csv.exists() and not OVERWRITE_LABELONLY:
        result["status"] = "SKIP_EXISTS"
        return result

    df = pd.read_csv(src_csv, low_memory=False)

    # Do not create new columns.
    # If the column set is valid but order differs, reorder using existing columns only.
    original_columns = list(df.columns)
    if original_columns != EXPECTED_COLUMNS:
        df = df[EXPECTED_COLUMNS].copy()
        result["reordered_columns"] = True

    before = len(df)
    active_num = pd.to_numeric(df["active"], errors="coerce").fillna(0).astype(int)
    active_rows_before = int((active_num == 1).sum())

    result["rows_before"] = int(before)
    result["active_rows_before"] = active_rows_before

    # T0/REST files remain full length because active==0 by design.
    if file_task == 0:
        out_df = df.copy()
        result["rows_after"] = int(len(out_df))
        result["rows_removed"] = 0
        result["status"] = "OK_T0_FULL_COPY"

    else:
        out_df = df[active_num == 1].copy()
        removed = before - len(out_df)

        result["rows_after"] = int(len(out_df))
        result["rows_removed"] = int(removed)
        result["status"] = "OK_ACTIVE_ONLY"

        if len(out_df) == 0:
            result["warning"] = "Non-T0 file has zero active rows after filtering"
            if SKIP_EMPTY_NONREST_OUTPUT:
                result["status"] = "SKIP_EMPTY_NONREST"
                return result

    # Save labelonly CSV.
    # This preserves exactly the EXPECTED_COLUMNS and never adds columns.
    out_df.to_csv(dest_csv, index=False)

    # Copy sidecars if available.
    if COPY_SIDECARS:
        copied = 0
        for sc in sidecars_for_label(src_csv):
            if sc.exists():
                dest_sc = dest_csv.parent / sc.name
                try:
                    shutil.copy2(sc, dest_sc)
                    copied += 1
                except Exception as e:
                    result["warning"] = (result["warning"] + " | " if result["warning"] else "") + f"sidecar copy failed: {sc.name}: {e}"
        result["sidecars_copied"] = copied

    return result


print("Phase 2B export function loaded.")


Phase 2B export function loaded.


In [ ]:
# ============================================================
# CELL 7 — Run Phase 2B for all subjects
# Purpose:
#   Export labelonly files from all valid Phase 2A label files.
# ============================================================

export_results = []

if RUN_ALL_SUBJECTS:
    print("=" * 100)
    print("RUNNING PHASE 2B FOR ALL SUBJECTS")
    print("=" * 100)
    print("Total Phase 2A label files to process:", len(label_files))

    for i, src_csv in enumerate(label_files, start=1):
        print("\n" + "-" * 100)
        print(f"[{i}/{len(label_files)}] {src_csv}")

        try:
            res = export_one_phase2b_labelonly(src_csv)
            export_results.append(res)

            print({
                "status": res["status"],
                "subject": res["subject"],
                "task": res["task"],
                "trial": res["trial"],
                "rows_before": res["rows_before"],
                "rows_after": res["rows_after"],
                "rows_removed": res["rows_removed"],
                "dest_csv": res["dest_csv"],
                "warning": res["warning"],
            })

        except Exception as e:
            err = traceback.format_exc(limit=2)
            print("❌ ERROR:", e)
            print(err)
            export_results.append({
                "src_csv": str(src_csv),
                "dest_csv": "",
                "subject": f"Sub-{get_subject_id_from_path(src_csv)}",
                "filename": src_csv.name,
                "status": "ERROR_EXCEPTION",
                "error": str(e),
                "warning": "",
                "task": None,
                "trial": None,
                "is_rest_task0": None,
                "rows_before": None,
                "rows_after": None,
                "rows_removed": None,
                "active_rows_before": None,
                "mode": None,
                "reordered_columns": None,
                "sidecars_copied": None,
            })

    export_df = pd.DataFrame(export_results)

    export_summary_path = AUDIT_DIR / "phase2B_export_summary.csv"
    export_df.to_csv(export_summary_path, index=False)

    print("\n" + "=" * 100)
    print("PHASE 2B EXPORT SUMMARY")
    print("=" * 100)
    print(export_df["status"].value_counts(dropna=False))
    print("\nSaved:", export_summary_path)

    # Active-row removal log
    filter_log_df = export_df[
        export_df["status"].isin(["OK_ACTIVE_ONLY", "OK_T0_FULL_COPY"]) |
        export_df["status"].astype(str).str.startswith("OK")
    ].copy()

    filter_log_path = AUDIT_DIR / "phase2B_active_zero_rows_removed.csv"
    filter_log_df.to_csv(filter_log_path, index=False)
    print("Saved row-filter log:", filter_log_path)

    # Invalid/skipped log
    invalid_df = export_df[~export_df["status"].astype(str).str.startswith("OK")].copy()
    invalid_path = AUDIT_DIR / "phase2B_invalid_or_skipped.csv"
    invalid_df.to_csv(invalid_path, index=False)
    print("Saved invalid/skipped log:", invalid_path)

    if len(invalid_df) > 0:
        print("\n⚠️ Some files were skipped or errored:")
        display(invalid_df[["subject", "filename", "status", "error", "warning"]].head(30))
    else:
        print("\n✅ No invalid/skipped files in Phase 2B export.")

else:
    print("RUN_ALL_SUBJECTS=False. No Phase 2B export was run.")
    export_df = pd.DataFrame()


In [ ]:
# ============================================================
# CELL 8 — Final verification of labelonly outputs
# Purpose:
#   Confirm downstream phases will use only labelonly outputs,
#   and the 104 missing Phase 1C files remain excluded.
# ============================================================

labelonly_files = sorted(
    ROOT_DIR.glob(f"Sub-*/{SYNC_SUBPATH}/{DEST_LABEL_DIRNAME}/{LABEL_GLOB}")
)

phase2a_files = sorted(
    ROOT_DIR.glob(f"Sub-*/{SYNC_SUBPATH}/{SRC_LABEL_DIRNAME}/{LABEL_GLOB}")
)

print("=" * 100)
print("FINAL PHASE 2B VERIFICATION")
print("=" * 100)
print("Phase 2A label files:", len(phase2a_files))
print("Phase 2B labelonly files:", len(labelonly_files))

verify_rows = []

for f in labelonly_files:
    try:
        header = read_header_only(f)
        df_small = pd.read_csv(f, low_memory=False)

        mode_char, file_task, file_trial = parse_task_trial_from_filename(f)
        subject_from_path = get_subject_id_from_path(f)

        errors = []
        warnings = []

        missing = [c for c in EXPECTED_COLUMNS if c not in header]
        extra = [c for c in header if c not in EXPECTED_SET]
        if missing:
            errors.append(f"missing={missing}")
        if extra:
            errors.append(f"extra={extra}")

        if header != EXPECTED_COLUMNS:
            warnings.append("column_order_differs")

        # Metadata checks if non-empty
        if len(df_small) > 0:
            sub_val, _ = safe_unique_int(df_small["subject_id"])
            task_val, _ = safe_unique_int(df_small["task"])
            trial_val, _ = safe_unique_int(df_small["trial"])

            if sub_val != subject_from_path:
                errors.append(f"subject mismatch csv={sub_val}, folder={subject_from_path}")
            if task_val != file_task:
                errors.append(f"task mismatch csv={task_val}, filename={file_task}")
            if trial_val != file_trial:
                errors.append(f"trial mismatch csv={trial_val}, filename={file_trial}")

            active_vals = sorted(pd.to_numeric(df_small["active"], errors="coerce").fillna(0).astype(int).unique().tolist())

            if file_task == 0:
                # REST/T0 file should stay full and should be active=0.
                if not set(active_vals).issubset({0}):
                    warnings.append(f"T0 has active values {active_vals}")
            else:
                # Non-T0 labelonly should contain active rows only, unless empty.
                if len(df_small) > 0 and not set(active_vals).issubset({1}):
                    errors.append(f"non-T0 labelonly contains non-active values {active_vals}")

        verify_rows.append({
            "file": str(f),
            "subject": f"Sub-{subject_from_path}",
            "filename": f.name,
            "n_rows": len(df_small),
            "n_cols": len(header),
            "task": file_task,
            "trial": file_trial,
            "status": "OK" if not errors else "ERROR",
            "errors": " | ".join(errors),
            "warnings": " | ".join(warnings),
        })

    except Exception as e:
        verify_rows.append({
            "file": str(f),
            "subject": None,
            "filename": f.name,
            "n_rows": None,
            "n_cols": None,
            "task": None,
            "trial": None,
            "status": "ERROR",
            "errors": str(e),
            "warnings": "",
        })

verify_df = pd.DataFrame(verify_rows)

verify_path = AUDIT_DIR / "phase2B_labelonly_final_verification.csv"
verify_df.to_csv(verify_path, index=False)

print("\nVerification status:")
print(verify_df["status"].value_counts(dropna=False))
print("\nSaved:", verify_path)

subject_counts = (
    verify_df.groupby("subject")["filename"]
    .count()
    .reset_index(name="phase2B_labelonly_files")
    .sort_values("subject")
)
subject_counts_path = AUDIT_DIR / "phase2B_labelonly_counts_by_subject.csv"
subject_counts.to_csv(subject_counts_path, index=False)

print("\nLabelonly files by subject:")
display(subject_counts)

# Show row counts by task
task_counts = (
    verify_df.groupby("task")["filename"]
    .count()
    .reset_index(name="files")
    .sort_values("task")
)
print("\nLabelonly file counts by task:")
display(task_counts)

# Check expected count
if len(labelonly_files) == len(phase2a_files) and len(labelonly_files) > 0:
    print("\n✅ Phase 2B output count matches Phase 2A label count.")
else:
    print("\n⚠️ Phase 2B output count does not match Phase 2A label count. Check skipped/error logs.")

if len(verify_df[verify_df["status"] != "OK"]) > 0:
    print("\n❌ Some labelonly outputs failed verification:")
    display(verify_df[verify_df["status"] != "OK"].head(30))
    raise RuntimeError("Stop: labelonly verification failed.")
else:
    print("\n✅ All labelonly outputs passed final verification.")

print("\nDownstream phases should read only this pattern:")
print(ROOT_DIR / "Sub-*" / SYNC_SUBPATH / DEST_LABEL_DIRNAME / LABEL_GLOB)

print("\n✅ Missing Phase 1C files remain excluded because Phase 2B used only existing Phase 2A labels.")


In [9]:
# ============================================================
# CELL 9 — Quick single-file inspection after Phase 2B
# Purpose:
#   Open one labelonly CSV and print columns/row counts.
# ============================================================

if len(labelonly_files) == 0:
    print("No labelonly files found.")
else:
    example = labelonly_files[0]

    # Prefer a non-T0 file with rows, if available
    for f in labelonly_files:
        _, task, _ = parse_task_trial_from_filename(f)
        try:
            n = len(pd.read_csv(f, nrows=5))
        except Exception:
            n = 0
        if task != 0 and n > 0:
            example = f
            break

    print("=" * 100)
    print("Example Phase 2B labelonly CSV:")
    print(example)
    print("=" * 100)

    ex_df = pd.read_csv(example, low_memory=False)

    print("Shape:", ex_df.shape)
    print("Columns:")
    for c in ex_df.columns:
        print(" -", c)

    print("\nMetadata:")
    print("subject_id:", ex_df["subject_id"].dropna().unique() if len(ex_df) else "EMPTY")
    print("task      :", ex_df["task"].dropna().unique() if len(ex_df) else "EMPTY")
    print("trial     :", ex_df["trial"].dropna().unique() if len(ex_df) else "EMPTY")

    if len(ex_df) > 0:
        print("\nactive value counts:")
        print(ex_df["active"].value_counts(dropna=False).sort_index())

        print("\nlabel_action value counts:")
        print(ex_df["label_action"].value_counts(dropna=False).sort_index())

    print("\n✅ Example labelonly inspection completed.")


Example Phase 2B labelonly CSV:
/home/tsultan1/BioRob/Human Subject Data/Sub-1/cleaned/synchronized_proper_lite_union_v3/labelonly/002_T516_synchronized_corrected_icml_consensus_labels.csv
Shape: (919, 45)
Columns:
 - subject_id
 - task
 - trial
 - Timestamp_seconds
 - Timestamp_ms
 - EMG_Ch1
 - EMG_Ch2
 - EMG_Ch3
 - EMG_Ch4
 - EEG_Ch1
 - EEG_Ch2
 - EEG_Ch3
 - EEG_Ch4
 - EEG_Ch5
 - EEG_Ch6
 - EEG_Ch7
 - EEG_Ch8
 - ET_GazeLeftx
 - ET_GazeLefty
 - ET_GazeRightx
 - ET_GazeRighty
 - ET_PupilLeft
 - ET_PupilRight
 - ET_ValidityLeftEye
 - ET_ValidityRightEye
 - ET_Blink
 - ET_Fixation
 - ET_Worn
 - ET_DistanceLeft
 - ET_DistanceRight
 - ET_GyroX
 - ET_GyroY
 - ET_GyroZ
 - ET_AccX
 - ET_AccY
 - ET_AccZ
 - ET_HeadRotationPitch
 - ET_HeadRotationYaw
 - ET_HeadRotationRoll
 - active_raw
 - active
 - active_prob
 - label_action
 - label_11
 - task_target

Metadata:
subject_id: [1]
task      : [5]
trial     : [16]

active value counts:
active
1    919
Name: count, dtype: int64

label_action value 